In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans

SEED = 1660

## Читаем данные

In [ ]:
df_train = pd.read_parquet("03_datasets/final_train.parquet")
df_test = pd.read_parquet("03_datasets/final_test.parquet")

In [ ]:
X_train = df_train.drop(columns=["GameId", "Elo"])
Y_train = df_train["Elo"]

X_test = df_test.drop(columns=["GameId", "Elo"])
Y_test = df_test["Elo"]

## Делаем PCA

In [ ]:
mean, std = X_train.mean(), X_train.std()
X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
pca.fit(X_train_scaled)
df_train_clust = pd.DataFrame(pca.transform(X_train_scaled))
df_test_clust = pd.DataFrame(pca.transform(X_test_scaled))
print(pca.explained_variance_ratio_ * 100)

## График

In [ ]:
df_train_clust["Y"] = Y_train.values

In [ ]:
cluster_model = MiniBatchKMeans(n_clusters=250, batch_size=20_000, random_state=SEED, n_init=5)
cluster_model.fit(df_train_clust[[0, 1]])
df_train_clust["cluster"] = cluster_model.predict(df_train_clust[[0, 1]])
df_test_clust["cluster"] = cluster_model.predict(df_test_clust[[0, 1]])

In [ ]:
coloring = (
    df_train_clust
    .groupby("cluster")
    .agg({"Y": "mean"})
    .squeeze().to_dict()
)

df_train_clust["color"] = df_train_clust["cluster"].map(coloring)
df_test_clust["color"] = df_test_clust["cluster"].map(coloring)

In [ ]:
fig = px.scatter(
    df_train_clust.sample(50_000),
    x=0,
    y=1,
    color="color",
#     color="cluster",
#     color="Y",
    color_continuous_scale=["darkred", "red", "orange", "yellow", "lime", "green"]
)

fig.data[0].marker.size=2.5

fig.update_layout(
    template="plotly_dark",
    height=950,
    width=950
)

fig.show()